# Data Loading & Preparation

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, StandardScaler

# We load two separate datasets:
# A. The original feature-engineered dataset (for Lasso/Ridge interpretation).
# B. The PCA-transformed dataset (provided by your group mate).

# --- A. Load Original Data ---
train_path_orig = 'feature-engineering-transform-train.csv'
test_path_orig = 'feature-engineering-transform-test.csv'

df_train_orig = pd.read_csv(train_path_orig)
df_test_orig = pd.read_csv(test_path_orig)

# Separate Features (X) and Target (y)
X_train_orig = df_train_orig.drop(columns=['label'], errors='ignore')
y_train = df_train_orig['label'] 
X_test_orig = df_test_orig.drop(columns=['label'], errors='ignore')
y_test = df_test_orig['label']

# --- B. Load Friend's PCA Data ---
train_path_pca = 'feature-engineering-pca-train.csv' 
test_path_pca = 'feature-engineering-pca-test.csv'

df_train_pca = pd.read_csv(train_path_pca)
df_test_pca = pd.read_csv(test_path_pca)

# Ensure to drop label if it exists in the PCA file
X_train_pca = df_train_pca.drop(columns=['label'], errors='ignore')
X_test_pca = df_test_pca.drop(columns=['label'], errors='ignore')

print(f"Original Data Shape: {X_train_orig.shape}")
print(f"PCA Data Shape:      {X_train_pca.shape}")

KeyError: 'label'

#  PREPROCESSOR 


In [ ]:
# We only define a preprocessor for the ORIGINAL data to fix assumptions.
# For PCA data, we do absolutely nothing (since PCA already transformed it).

# Detect Column Types automatically
binary_cols = [col for col in X_train_orig.columns if X_train_orig[col].nunique() <= 2]
numerical_cols = [col for col in X_train_orig.columns if col not in binary_cols]

# Define Pipeline A: Interpretable Fixer
# 1. Continuous Vars: Apply Yeo-Johnson (Fixes Linearity & Outliers)
# 2. Binary Vars: Passthrough (Don't touch them)
prep_interpretable = ColumnTransformer([
    ('num', PowerTransformer(method='yeo-johnson'), numerical_cols), 
    ('cat', 'passthrough', binary_cols) 
])

Detected Binary Cols: 24
Detected Numerical Cols: 22


#  MODEL DEFINITIONS


In [ ]:
# --- Group 1: Models for Original Data ---
models_orig = {
    "Base (No Reg)": LogisticRegression(penalty='l2', C=1e9, max_iter=3000, random_state=42, class_weight='balanced'),
    "Lasso (L1)":    LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear', max_iter=3000, scoring='f1', random_state=42, class_weight='balanced'),
    "Ridge (L2)":    LogisticRegressionCV(cv=5, penalty='l2', max_iter=3000, scoring='f1', random_state=42, class_weight='balanced'),
    "ElasticNet":    LogisticRegressionCV(cv=5, penalty='elasticnet', solver='saga', l1_ratios=[0.5], max_iter=3000, scoring='f1', random_state=42, class_weight='balanced')
}

# --- Group 2: Models for PCA Data ---
models_pca = {
    # Direct models, no pipelines attached
    "PCA + Base": LogisticRegression(penalty='l2', C=1e9, max_iter=3000, class_weight='balanced', random_state=42),
    "PCA + L1":   LogisticRegressionCV(cv=5, penalty='l1', solver='liblinear', scoring='f1', class_weight='balanced', random_state=42)
}


--- CHECKING LINEARITY ASSUMPTION ---


/Users/akkus/Documents/GitHub/Phishing-URL--Website--Detection/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


<Figure size 1200x1000 with 0 Axes>

NOTE: If plots are curved, our Pipeline's 'PowerTransformer' will fix this automatically.


#  TRAINING LOOP


In [ ]:
results = []

print("\n--- Starting Training Process ---")

# --- Loop 1: Original Data (With Preprocessing Pipeline) ---
print(">> Training Models on ORIGINAL Data (with Yeo-Johnson Fix)...")
for name, model in models_orig.items():
    # Construct Pipeline
    clf = Pipeline([
        ('prep', prep_interpretable),
        ('model', model)
    ])
    
    # Train
    clf.fit(X_train_orig, y_train)
    
    # Predict
    y_pred = clf.predict(X_test_orig)
    y_prob = clf.predict_proba(X_test_orig)[:, 1]
    
    # Metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    results.append({
        "Dataset": "Original",
        "Model Strategy": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "Sensitivity": recall_score(y_test, y_pred),
        "Specificity": tn / (tn + fp),
        "Kappa": cohen_kappa_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

# --- Loop 2: PCA Data (DIRECTLY - NO SCALING) ---
print(">> Training Models on PCA Data (Directly - No Scaling)...")
for name, model in models_pca.items():
    
    # DIRECT FIT: No Pipeline, No Scaling.
    model.fit(X_train_pca, y_train)
    
    # Predict
    y_pred = model.predict(X_test_pca)
    y_prob = model.predict_proba(X_test_pca)[:, 1]
    
    # Metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    results.append({
        "Dataset": "PCA Transformed",
        "Model Strategy": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "Sensitivity": recall_score(y_test, y_pred),
        "Specificity": tn / (tn + fp),
        "Kappa": cohen_kappa_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    })

#  FINAL REPORTING


In [ ]:
df_results = pd.DataFrame(results)
# Sort by F1-Score (Highest on top)
df_results = df_results.sort_values(by="F1-Score", ascending=False)

print("\n" + "="*50)
print("FINAL MODEL COMPARISON TABLE")
print("="*50)
print(df_results.to_string(index=False))

# --- Coefficient Interpretation (Original Data Only) ---
print("\n" + "="*50)
print("COEFFICIENT INTERPRETATION (Best Original Model)")
print("="*50)

# Identify the best original model
best_orig_model_name = df_results[df_results['Dataset'] == 'Original'].iloc[0]['Model Strategy']
print(f"Best Interpretable Model: {best_orig_model_name}")

# Re-train just that model to get coefficients
best_model_instance = models_orig[best_orig_model_name]
pipeline_final = Pipeline([('prep', prep_interpretable), ('model', best_model_instance)])
pipeline_final.fit(X_train_orig, y_train)

# Extract and Print Coefficients
if hasattr(pipeline_final.named_steps['model'], 'coef_'):
    # Reconstruct feature names (Numerical first, then Binary)
    feature_names_reordered = numerical_cols + binary_cols
    coefs = pipeline_final.named_steps['model'].coef_[0]
    
    coef_df = pd.DataFrame({'Feature': feature_names_reordered, 'Coefficient': coefs})
    coef_df['Abs_Coef'] = coef_df['Coefficient'].abs()
    
    print(coef_df.sort_values(by='Abs_Coef', ascending=False).head(15))